In [1]:
! pip install -U langchain_community tiktoken langchain-openai langchain-cohere langchainhub chromadb langchain langgraph  tavily-python

  Using cached langchain_openai-1.1.10-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain-1.2.10-py3-none-any.whl.metadata (5.7 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached ty

In [4]:
import getpass
import os


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("OPENAI_API_KEY")
# _set_env("COHERE_API_KEY")
_set_env("TAVILY_API_KEY")

OPENAI_API_KEY:  ········
TAVILY_API_KEY:  ········


In [8]:
# from langchain_community.document_loaders import WebBaseLoader
help(Literal)

Help on _TypedCacheSpecialForm in module typing:

Literal = typing.Literal
    Special typing form to define literal types (a.k.a. value types).

    This form can be used to indicate to type checkers that the corresponding
    variable or function parameter has a value equivalent to the provided
    literal (or one of several literals)::

        def validate_simple(data: Any) -> Literal[True]:  # always returns True
            ...

        MODE = Literal['r', 'rb', 'w', 'wb']
        def open_helper(file: str, mode: MODE) -> str:
            ...

        open_helper('/some/path', 'r')  # Passes type check
        open_helper('/other/path', 'typo')  # Error in type checker

    Literal[...] cannot be subclassed. At runtime, an arbitrary value
    is allowed as type argument to Literal[...], but type checkers may
    impose restrictions.



In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Initialize embeddings
embeddings = OpenAIEmbeddings()

# URLs to load
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents
docs = []
for url in urls:
    loader = WebBaseLoader(url)
    docs.extend(loader.load())

# Split documents
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=50
)

splits = text_splitter.split_documents(docs)

# Create vector database
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="rag-chroma",
    # persist_directory="./chroma_db"
)

# Create retriever
retriever = vectorstore.as_retriever()

print(f"Successfully indexed {len(splits)} chunks.")

Successfully indexed 91 chunks.


In [3]:
! pip install langchain-community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)


In [6]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["vectorstore", "web_search"] = Field(
        description="Given a user question choose to route it to web search or a vectorstore.",
    )

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
structured_llm_router = llm.with_structured_output(RouteQuery)

system = """You are an expert at routing a user question to a vectorstore or web search.
The vectorstore contains documents related to agents, prompt engineering, and adversarial attacks.
Use the vectorstore for questions on these topics. Otherwise, use web-search."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}")
    ]
)
question_router = route_prompt | structured_llm_router
print(question_router.invoke(
    {"question": "Who will the Bears draft first in the NFL draft?"}
))
print(question_router.invoke({"question": "What are the types of agent memory?"}))

D:\Download_New\anaconda\envs\ai_env\Lib\site-packages\langchain_openai\chat_models\base.py:2214: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


datasource='web_search'
datasource='vectorstore'


In [7]:
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

system = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prompt=ChatPromptTemplate.from_messages(
    [
        ("system", system),
         ("human", "Retrieved document: \n\n {document} \n\n User question: {question}")
    ]
)
retrieval_grader = grade_prompt | structured_llm_grader

question = "agent memeory"
docs = retriever.invoke(question)
doc_txt=docs[1].page_content
print(retrieval_grader.invoke({"question": question, "document":doc_txt}))


binary_score='yes'


In [8]:
from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser

prompt=hub.pull('rlm/rag-prompt')

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

def format_docs(docs):
    return "\n\n".join(docs.page_content for doc in docs)
    
rag_chain  = prompt | llm | StrOutputParser()

generation = rag_chain.invoke({"context": docs, "question": question})
print(generation)

Agent memory in a LLM-powered autonomous agent system consists of short-term memory for in-context learning and long-term memory for retaining and recalling information over extended periods. The external memory can support fast retrieval and alleviate the restriction of finite attention span. It is crucial for the agent's problem-solving capabilities and overall performance.


In [18]:
! pip install langchainhub

In [9]:
class GraderHallucination(BaseModel):
    binary_score: str = Field()

llm = ChatOpenAI(model = "gpt-3.5-turbo", temperature = 0)

structured_llm_grader = llm.with_structured_output(GraderHallucination)

system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
     Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""

hallucination_prompt  = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "set of facts: \n\n {document} \n\n llm generation: {generation}")
    ]
)

hallucination_grader = hallucination_prompt  | structured_llm_grader
hallucination_grader.invoke({"document": docs, "generation": generation})

D:\Download_New\anaconda\envs\ai_env\Lib\site-packages\langchain_openai\chat_models\base.py:2214: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


GraderHallucination(binary_score='yes')

In [10]:
# Data model
class GradeAnswer(BaseModel):
    """Binary score to assess answer addresses question."""

    binary_score: str = Field(
        description="Answer addresses the question, 'yes' or 'no'"
    )


# LLM with function call
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeAnswer)

# Prompt
system = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question."""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structured_llm_grader
answer_grader.invoke({"question": question, "generation": generation})

D:\Download_New\anaconda\envs\ai_env\Lib\site-packages\langchain_openai\chat_models\base.py:2214: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo-0125 since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


GradeAnswer(binary_score='yes')

In [11]:
llm = ChatOpenAI(model = "gpt-3.5-turbo", temperature=0)

system = """You a question re-writer that converts an input question to a better version that is optimized \n 
     for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning."""

re_write_prompt = ChatPromptTemplate.from_messages(
    
        [
            ("system", system),
            ("human", "Here is the initial question: \n\n {question} \n Formulate an improved question.")
        ]
    
)

question_rewriter = re_write_prompt | llm | StrOutputParser()
question_rewriter.invoke({"question": question})

"What is the role of memory in an agent's functioning?"

In [12]:
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool=TavilySearchResults(k=3)

C:\Users\Narisetti Chand\AppData\Local\Temp\ipykernel_21804\277589992.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool=TavilySearchResults(k=3)


In [13]:
from typing import List
from typing_extensions import TypedDict

class GraphState(TypedDict):
    question: str
    generation: str
    documents: List[str]

In [15]:
from langchain_core.documents import Document

def retrieve(state):
    print("---Retrieve---")
    question = state["question"]
    documents=retriever.invoke(questions)
    return {"documents": documents, "question": question}

def generate(state):
    print("---generate---")
    questin=state["question"]
    documents=state["documents"]
    generation=rag_chain.invoke({"context": documents, "question": question})
    return {"documents": documents, "question": question, "generation": generation}

def grade_documents(state):
    question=state["question"]
    documents=state["documents"]

    filtered_docs = []
    for d in documents:
        score=retrieval_grader.invoke({"question": question, "documents": d.page_content})
        grade=score.binary_score
        if grade == "yes":
            print("----garde: document relevent")
            filtered_docs.append(d)
        else:
            print("---grade documents not relevent")
    return {"documents": filtered_docs, "question": question}

def transform_query(state):
    question=state["question"]
    documents=state["documents"]
    better_question=question_rewriter.invoke({"question": question})
    return {"documents": documents, "question": better_question}

def web_search(state):
    print("----web searching")
    question=state["question"]
    web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in docs])
    web_results = Document(page_content=web_results)
    return {"documents": web_results, "question": question}

def route_question(state):
    print("---route question")
    question=state["question"]
    source=question_router.invoke({"question": question})
    if source.datasource == "web_search":
        print("---ROUTE QUESTION TO WEB SEARCH---")
        return "web_search"
    elif source.datasource == "vectorstore":
        print("---ROUTE QUESTION TO RAG---")
        return "vectorstore"
        
def decide_to_generate(state):
    print("---ASSESS GRADED DOCUMENTS---")
    state["question"]
    filtered_documents=state["documents"]
    if not filtered_documents:
        print("---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---")
        return "transform_query"
    else:
        print("--decision: generate")
        return "generate"

def grade_generation_v_documents_and_question(state):
    print("check hallucinations")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]
    hallucination_grader.invoke(
        {"documents": documents, "generation": generation}
    )
    grade = score.binary_score
    if grade == "yes":
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")
        # Check question-answering
        print("---GRADE GENERATION vs QUESTION---")
        answer_grader.invoke({"question": question, "generation": generation})
        grade = score.binary_score
        if grade == "yes":
            rint("---DECISION: GENERATION ADDRESSES QUESTION---")
            return "useful"
        else:
            print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION---")
            return "not useful"
    else:
        print("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"


In [2]:
! pip install langgraph

In [17]:
from langgraph.graph import END, StateGraph, START

workflow=StateGraph(GraphState)

workflow.add_node("web_search", web_search)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents",grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query",transform_query)

workflow.add_conditional_edges(START, route_question, {
    "web_search": "web_search",
    "vectorstore": "retrieve",
})
workflow.add_edge("web_search", "generate")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges("grade_documents", decide_to_generate,
                              {
                                  "transform_query":"transform_query",
                                  "generate": "generate"
                              })
workflow.add_edge("transform_query", "retrieve")
workflow.add_conditional_edges(
    "generate",
    grade_generation_v_documents_and_question,
    {
        "not supported": "generate",
        "useful":  END,
        "not useful": "transform_query",
    }
)

app = workflow.compile()